data extraction

Here is a complete breakdown of every feature extracted by the pipeline, including the mathematical formulas used to calculate them and their significance for your research paper.

I have grouped them into four logical categories: **Size**, **Shape**, **Optical (Intensity)**, and **Surface Texture (GLCM)**.

---

### 1. Size & Dimensional Metrics

These metrics define the physical footprint of the particle. Because we applied the `PIXEL_SCALE` (0.1116 µm), these values represent true physical dimensions.

* **Area_Physical (µm²)**
* **Formula:** $\text{Pixel Count} \times (\text{Pixel Scale})^2$
* **Significance:** The foundational metric for determining particle size distribution. In light microscopy, this represents the 2D projected footprint of the particle.


* **Perimeter_Physical (µm)**
* **Formula:** The continuous line length around the particle boundary $\times \text{Pixel Scale}$.
* **Significance:** Helps identify edge complexity. A highly jagged particle will have a much larger perimeter than a smooth particle of the exact same area.


* **Equivalent Circular Diameter / ECD (µm)**
* **Formula:** 
$$ECD = 2 \times \sqrt{\frac{Area}{\pi}}$$


* **Significance:** This is a standard reporting metric in materials science. It answers the question: *"If this irregularly shaped particle were a perfect circle, what would its diameter be?"* It allows you to plot standard size histograms (like D10, D50, D90) regardless of the particle's actual shape.


* **Major & Minor Axis (µm)**
* **Formula:** The longest (Major) and shortest (Minor) dimensions of the bounding box fitted around the particle.
* **Significance:** Useful for defining the maximum span of elongated particles (like fibers, flakes, or rods) where ECD might be misleading.



---

### 2. Shape & Morphology Metrics (Dimensionless)

These metrics describe *how* a particle is shaped, independent of its size. Because they are ratios, the physical scale cancels out.

* **Aspect Ratio**
* **Formula:** 
$$Aspect\ Ratio = \frac{Major\ Axis}{Minor\ Axis}$$


* **Significance:** Measures elongation. An aspect ratio of $1.0$ means the particle is perfectly symmetric (like a square or circle). High values indicate needle-like or rod-like particles. This heavily influences how particles flow and pack together in powder form.


* **Circularity**
* **Formula:** 
$$Circularity = \frac{4\pi \times Area}{Perimeter^2}$$


* **Significance:** Measures how close the 2D shape is to a perfect mathematical circle (ranges from 0 to 1). Highly circular particles (values > 0.8) usually indicate a gas-atomized or melted formation process. Low values indicate crushed, milled, or highly agglomerated particles.


* **Solidity**
* **Formula:** 
$$Solidity = \frac{Area}{Area_{Convex\ Hull}}$$


* *(Note: The "Convex Hull" is the shape you would get if you stretched a rubber band around the outside points of the particle).*
* **Significance:** Measures overall macro-roughness or "spikiness." A solid, blocky particle has a Solidity of 1. A particle with deep indentations, arms, or C-shapes (like a red blood cell or a highly porous agglomerate) will have a low solidity.


* **Convexity**
* **Formula:** 
$$Convexity = \frac{Perimeter_{Convex\ Hull}}{Perimeter}$$


* **Significance:** Measures edge micro-roughness. While Solidity looks at missing *area*, Convexity looks at the *edge path*. A particle can be generally round (high solidity) but have a very bumpy, sandpaper-like surface (low convexity). This is highly relevant for SEM imaging.



---

### 3. Optical & Compositional Metrics

These metrics look at the pixel grayscale values (0 = pure black, 255 = pure white) inside the particle boundary.

* **Intensity_Mean**
* **Formula:** 
$$\mu = \frac{\sum (Pixel\ Values)}{\text{Number of Pixels}}$$


* **Significance:** In light microscopy, this indicates translucency or material density (darker particles might be thicker or made of a different phase). In SEM (BSE mode), brighter mean intensities explicitly indicate elements with a higher atomic number (Z-contrast).


* **Intensity_StdDev**
* **Formula:** Standard deviation ($\sigma$) of the pixel values.
* **Significance:** Measures internal uniformity. A low standard deviation means the particle is a solid, uniform color/material. A high standard deviation means the particle is multi-phase, has varied thickness, or has heavy internal shadows.



---

### 4. Surface Texture (GLCM) Metrics

The Gray-Level Co-occurrence Matrix (GLCM) maps how often pairs of pixels with specific grayscale values occur next to each other. These are the most advanced metrics in your pipeline and are incredibly powerful for SEM topography.

* **Texture_Contrast & Texture_Dissimilarity**
* **Concept:** Both measure the local variations in grayscale. Contrast weighs larger differences more heavily (squared differences).
* **Significance:** High contrast indicates sharp boundaries and steep topographical changes (e.g., deep pores or sharp crystalline facets in SEM). Low contrast indicates a smooth, flat surface.


* **Texture_Homogeneity**
* **Concept:** The inverse of contrast. Measures how similar neighboring pixels are.
* **Significance:** High homogeneity means the particle surface is very smooth and uniform, with gradual or no changes in topography/color.


* **Texture_Energy & Texture_ASM (Angular Second Moment)**
* **Concept:** ASM is the sum of squared elements in the GLCM; Energy is the square root of ASM.
* **Significance:** Measures textural uniformity or order. If a particle has a repeating, predictable surface pattern (like a crystalline grid or uniform machining marks), Energy will be high. Random, chaotic, and messy surfaces have low Energy.


* **Texture_Correlation**
* **Concept:** Measures how correlated a pixel is to its neighbor over the whole particle.
* **Significance:** Identifies directional textures. If your particles have striations, scratches, or layered steps on their surface, the correlation will be higher along the angle of those scratches.

In [ ]:
import torch
import torchvision
import cv2
import numpy as np
import os
import glob
import pandas as pd

from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator

from skimage.feature import graycoprops

# ============================================================
# SETTINGS
# ============================================================

MODEL_PATH = r"E:\Prithu\Sustain Image with COCO JSON\acc\mask_rcnn_particle_modelacc.pth"
INPUT_FOLDER = r"E:\Prithu\Sustain Image with COCO JSON\acc\images"
OUTPUT_FOLDER = r"E:\Prithu\Sustain Image with COCO JSON\Publication_Results_new_acc"

EXCEL_FOLDER = os.path.join(OUTPUT_FOLDER, "Excel_Data")
IMAGE_FOLDER = os.path.join(OUTPUT_FOLDER, "Visualized_Images")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 3
SCORE_THRESHOLD = 0.10
MAX_OVERLAP_ALLOWED = 0.10

# Calibration: 1 pixel = 1.11 physical units (e.g., microns)
PIXEL_SCALE = 0.1116

# ============================================================
# TEXTURE (GLCM) HELPER FUNCTIONS
# ============================================================

def quantize_image(image, mask, glcm_gray_levels=32):
    """Quantize image intensities within the mask to integer gray levels."""
    values = image[mask]
    if len(values) == 0:
        return np.zeros_like(image, dtype=np.uint8)

    min_val, max_val = np.nanmin(values), np.nanmax(values)
    if max_val == min_val:
        return np.zeros_like(image, dtype=np.uint8)

    scaled = np.clip((image - min_val) / (max_val - min_val), 0, 1)
    return np.floor(scaled * (glcm_gray_levels - 1)).astype(np.uint8)

def masked_graycomatrix(image, mask, glcm_gray_levels=32, distances=(1,), angles=(0, np.pi/4, np.pi/2, 3*np.pi/4)):
    """Compute a masked gray-level co-occurrence matrix."""
    image = image.astype(np.int32)
    mask = mask.astype(bool)
    
    glcm = np.zeros((glcm_gray_levels, glcm_gray_levels, len(distances), len(angles)), dtype=np.float64)
    rows, cols = image.shape

    for d_idx, distance in enumerate(distances):
        for a_idx, angle in enumerate(angles):
            dr, dc = int(round(np.sin(angle) * distance)), int(round(np.cos(angle) * distance))
            for r in range(rows):
                r2 = r + dr
                if r2 < 0 or r2 >= rows: continue
                for c in range(cols):
                    c2 = c + dc
                    if c2 < 0 or c2 >= cols: continue
                    
                    if mask[r, c] and mask[r2, c2]:
                        i, j = image[r, c], image[r2, c2]
                        if 0 <= i < glcm_gray_levels and 0 <= j < glcm_gray_levels:
                            glcm[i, j, d_idx, a_idx] += 1
                            glcm[j, i, d_idx, a_idx] += 1

    total = glcm.sum(axis=(0, 1), keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(total > 0, glcm / total, 0)

def extract_texture_features(intensity_crop, mask_crop, glcm_gray_levels=32):
    """Compute Haralick-style GLCM texture features for one cropped object."""
    feature_names = ["contrast", "dissimilarity", "homogeneity", "ASM", "energy", "correlation"]
    
    if mask_crop.sum() < 4: 
        return {f"Texture_{name}": np.nan for name in feature_names}

    quantized = quantize_image(intensity_crop, mask=mask_crop, glcm_gray_levels=glcm_gray_levels)
    glcm = masked_graycomatrix(quantized, mask_crop, glcm_gray_levels=glcm_gray_levels)

    features = {}
    for name in feature_names:
        value = graycoprops(glcm, name)
        features[f"Texture_{name}"] = np.nanmean(value)

    return features

# ============================================================
# CREATE MODEL
# ============================================================

def get_model(num_classes):
    model = maskrcnn_resnet50_fpn(weights=None)
    anchor_sizes = ((16,), (32,), (64,), (128,), (256,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * 5

    model.rpn.anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, num_classes)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes)

    model.roi_heads.detections_per_img = 1000
    model.roi_heads.score_thresh = 0.05
    return model

# ============================================================
# MAIN INFERENCE
# ============================================================

@torch.no_grad()
def batch_inference():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(EXCEL_FOLDER, exist_ok=True)
    os.makedirs(IMAGE_FOLDER, exist_ok=True)

    model = get_model(NUM_CLASSES)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    image_paths = [p for p in glob.glob(os.path.join(INPUT_FOLDER, "*.*")) if p.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))]

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    for img_path in image_paths:
        print(f"\nProcessing: {os.path.basename(img_path)}")

        img_bgr = cv2.imread(img_path)
        if img_bgr is None: continue
        h, w = img_bgr.shape[:2]

        # Convert to Grayscale for Intensity/Texture calculations
        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)

        # Preprocess for model
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_norm = (img_rgb.astype(np.float32) / 255.0 - mean) / std
        img_tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).float().to(DEVICE)

        # Model Prediction
        prediction = model([img_tensor])[0]
        masks = prediction["masks"].cpu().numpy()
        scores = prediction["scores"].cpu().numpy()

        image_results = []
        occupied_mask = np.zeros((h, w), dtype=np.uint8)
        combined_colors = np.zeros_like(img_bgr, dtype=np.uint8)
        boundary_canvas = img_bgr.copy()
        found_count = 0

        for i in range(len(scores)):
            if scores[i] < SCORE_THRESHOLD: continue

            raw_mask = (masks[i, 0] > 0.5).astype(np.uint8)
            raw_area = np.sum(raw_mask)
            if raw_area < 50: continue

            overlap_area = np.sum(cv2.bitwise_and(raw_mask, occupied_mask))
            if (overlap_area / raw_area) > MAX_OVERLAP_ALLOWED: continue

            clean_mask = cv2.bitwise_and(raw_mask, raw_mask, mask=cv2.bitwise_not(occupied_mask))
            kernel = np.ones((3, 3), np.uint8)
            clean_mask = cv2.erode(clean_mask, kernel, iterations=1)
            
            if np.sum(clean_mask) < 20: continue
            
            found_count += 1
            occupied_mask = cv2.bitwise_or(occupied_mask, clean_mask)

            contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if len(contours) == 0: continue
            cnt = max(contours, key=cv2.contourArea)

            # ====================================================
            # MORPHOLOGY (With Physical Scaling)
            # ====================================================
            area_px = cv2.contourArea(cnt)
            perimeter_px = cv2.arcLength(cnt, True)
            
            # Physical Scaled metrics
            area_phys = area_px * (PIXEL_SCALE ** 2)
            perimeter_phys = perimeter_px * PIXEL_SCALE

            rect = cv2.minAreaRect(cnt)
            (center), (width, height), angle = rect
            major_axis_phys = max(width, height) * PIXEL_SCALE
            minor_axis_phys = min(width, height) * PIXEL_SCALE

            aspect_ratio = major_axis_phys / (minor_axis_phys + 1e-6)
            circularity = (4 * np.pi * area_phys) / (perimeter_phys**2 + 1e-6)
            ecd_phys = 2 * np.sqrt(area_phys / np.pi)

            # Hull Metrics (Solidity & Convexity)
            hull = cv2.convexHull(cnt)
            hull_area_px = cv2.contourArea(hull)
            hull_peri_px = cv2.arcLength(hull, True)
            
            solidity = area_px / (hull_area_px + 1e-6) # Dimensionless
            convexity = hull_peri_px / (perimeter_px + 1e-6) # Dimensionless

            is_circular = "Yes" if (0.75 <= circularity <= 1.2 and aspect_ratio <= 1.33) else "No"

            # ====================================================
            # INTENSITY & TEXTURE (GLCM)
            # ====================================================
            # Bounding box for efficient cropping
            x, y, bw, bh = cv2.boundingRect(cnt)
            
            # Extract pixels belonging ONLY to this particle
            particle_pixels = img_gray[clean_mask > 0]
            mean_intensity = np.mean(particle_pixels) if len(particle_pixels) > 0 else 0
            std_intensity = np.std(particle_pixels) if len(particle_pixels) > 0 else 0
            
            # Crop for GLCM
            crop_gray = img_gray[y:y+bh, x:x+bw]
            crop_mask = clean_mask[y:y+bh, x:x+bw] > 0
            texture_features = extract_texture_features(crop_gray, crop_mask)

            # ====================================================
            # SAVE RESULTS
            # ====================================================
            particle_data = {
                "Particle_No": found_count,
                "Confidence": round(float(scores[i]), 3),
                "Area_Physical": round(area_phys, 2),
                "Perimeter_Physical": round(perimeter_phys, 2),
                "ECD_Physical": round(ecd_phys, 2),
                "Major_Axis_Physical": round(major_axis_phys, 2),
                "Minor_Axis_Physical": round(minor_axis_phys, 2),
                "Aspect_Ratio": round(aspect_ratio, 3),
                "Circularity": round(circularity, 3),
                "Solidity": round(solidity, 3),
                "Convexity": round(convexity, 3),
                "Orientation_Angle_deg": round(angle, 2),
                "Intensity_Mean": round(mean_intensity, 2),
                "Intensity_StdDev": round(std_intensity, 2),
                "Is_Circular": is_circular
            }
            # Add GLCM features dynamically
            particle_data.update({k: round(v, 4) for k, v in texture_features.items()})
            
            image_results.append(particle_data)

            # ====================================================
            # VISUALIZATION
            # ====================================================
            color = [int(c) for c in np.random.randint(60, 255, 3)]
            combined_colors[clean_mask > 0] = color
            cv2.drawContours(boundary_canvas, [cnt], -1, color, 2)

            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cX, cY = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                cv2.putText(boundary_canvas, str(found_count), (cX, cY), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # ====================================================
        # SAVE EXCEL & IMAGES
        # ====================================================
        if len(image_results) > 0:
            df = pd.DataFrame(image_results)
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            excel_path = os.path.join(EXCEL_FOLDER, f"{base_name}_particle_metrics.xlsx")
            df.to_excel(excel_path, index=False)
            print(f"Excel saved: {excel_path}")

        final_result = cv2.addWeighted(boundary_canvas, 0.7, combined_colors, 0.3, 0)
        output_img_path = os.path.join(IMAGE_FOLDER, f"visualized_{os.path.basename(img_path)}")
        cv2.imwrite(output_img_path, final_result)
        print(f"Particles detected: {found_count}")

if __name__ == "__main__":
    batch_inference()

train

In [ ]:
import os
import json
import numpy as np
import cv2
import torch
import torchvision
import multiprocessing
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Prevent OpenCV from deadlocking PyTorch
cv2.setNumThreads(0)

# ---------------------------------------------------------
# 1. DATA AUGMENTATION (Cleaned of all warnings)
# ---------------------------------------------------------
def get_train_transforms():
    return A.Compose([
        # Ensure dimensions are multiples of 32
        A.PadIfNeeded(min_height=None, min_width=None, pad_height_divisor=32, pad_width_divisor=32, border_mode=cv2.BORDER_CONSTANT, fill=0),
        
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        
        # Updated Affine parameters to remove UserWarnings
        A.Affine(
            translate_percent={"x": (0.1, 0.1), "y": (0.1, 0.1)}, 
            scale=(0.85, 1.15), 
            rotate=(-30, 30), 
            p=0.5, 
            interpolation=cv2.INTER_LINEAR
        ),
        
        A.OneOf([
            A.RandomBrightnessContrast(p=1),
            A.HueSaturationValue(p=1),
            A.GaussNoise(p=1),
        ], p=0.4),
        
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_area=8, min_visibility=0.1))

def get_val_transforms():
    return A.Compose([
        A.PadIfNeeded(min_height=None, min_width=None, pad_height_divisor=32, pad_width_divisor=32, border_mode=cv2.BORDER_CONSTANT, fill=0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# ---------------------------------------------------------
# 2. DATASET CLASS
# ---------------------------------------------------------
class CocoDataset(Dataset):
    def __init__(self, img_dir, ann_file, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        with open(ann_file) as f:
            coco = json.load(f)

        self.images = {img['id']: img for img in coco['images']}
        self.cat_map = {cat['id']: i+1 for i, cat in enumerate(coco['categories'])}
        self.img_to_anns = {}
        for ann in coco['annotations']:
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)
        self.ids = list(self.images.keys())

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.images[img_id]
        filename = info.get('file_name', info.get('image'))
        
        path = os.path.join(self.img_dir, filename)
        img = cv2.imread(path)
        if img is None: raise FileNotFoundError(f"Missing: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        anns = self.img_to_anns.get(img_id, [])
        boxes, labels, masks = [], [], []

        for ann in anns:
            x, y, bw, bh = ann['bbox']
            x_min, y_min, x_max, y_max = max(0, x), max(0, y), min(w, x + bw), min(h, y + bh)
            if (x_max - x_min) <= 1 or (y_max - y_min) <= 1: continue

            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(self.cat_map[ann['category_id']])
            mask = np.zeros((h, w), dtype=np.uint8)
            for seg in ann['segmentation']:
                pts = np.array(seg, dtype=np.int32).reshape(-1, 2)
                cv2.fillPoly(mask, [pts], 1)
            masks.append(mask)

        if self.transforms:
            transformed = self.transforms(image=img, bboxes=boxes, masks=masks, labels=labels)
            img, boxes, masks, labels = transformed['image'], transformed['bboxes'], transformed['masks'], transformed['labels']

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
            "masks": torch.as_tensor(np.stack(masks), dtype=torch.uint8) if masks else torch.zeros((0, img.shape[1], img.shape[2]), dtype=torch.uint8),
            "image_id": torch.tensor([img_id])
        }
        return img, target

    def __len__(self):
        return len(self.ids)

# ---------------------------------------------------------
# 3. UTILS & EVALUATION
# ---------------------------------------------------------
class EarlyStopping:
    def __init__(self, patience=35, min_delta=0.001):
        self.patience, self.min_delta, self.counter, self.best_score, self.early_stop = patience, min_delta, 0, None, False
    def __call__(self, val_iou):
        if self.best_score is None: self.best_score = val_iou
        elif val_iou < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else: self.best_score, self.counter = val_iou, 0

def collate_fn(batch):
    return tuple(zip(*batch))

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_iou = []
    for imgs, targets in loader:
        imgs = [img.to(device) for img in imgs]
        preds = model(imgs)
        for i in range(len(imgs)):
            if len(targets[i]['masks']) == 0 or len(preds[i]['masks']) == 0:
                all_iou.append(0.0)
                continue
            t_mask = (targets[i]['masks'].to(device).sum(0) > 0).float()
            p_mask = (preds[i]['masks'].squeeze(1).sum(0) > 0.5).float()
            inter = (t_mask * p_mask).sum().item()
            union = t_mask.sum().item() + p_mask.sum().item() - inter
            all_iou.append(inter / (union + 1e-6))
    return np.mean(all_iou) if all_iou else 0.0

# ---------------------------------------------------------
# 4. TRAINING ENGINE
# ---------------------------------------------------------
def train(model, train_loader, val_loader, device, epochs):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    early_stopper = EarlyStopping(patience=35)

    print(f"Training on: {device}")
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for imgs, targets in train_loader:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())
            optimizer.zero_grad(); losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step(); epoch_loss += losses.item()

        val_iou = evaluate(model, val_loader, device)
        scheduler.step(val_iou)
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss/len(train_loader):.4f} | Val IoU: {val_iou:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        early_stopper(val_iou)
        if early_stopper.early_stop: 
            print(f"Early Stopping triggered at epoch {epoch+1}")
            break

# ---------------------------------------------------------
# 5. MAIN
# ---------------------------------------------------------
def main():
    base_dir = r"E:\Prithu\Sustain Image with COCO JSON"
    img_dir, ann_file = os.path.join(base_dir, "images"), os.path.join(base_dir, "sustain_fixed.json")
    
    # Target GPU 1
    device = torch.device("cuda:1" if torch.cuda.device_count() >= 2 else "cuda:0")

    full_dataset = CocoDataset(img_dir, ann_file)
    train_ds, val_ds = random_split(full_dataset, [17, 4], generator=torch.Generator().manual_seed(42))
    train_ds.dataset.transforms, val_ds.dataset.transforms = get_train_transforms(), get_val_transforms()

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

    num_classes = len(full_dataset.cat_map) + 1
    model = maskrcnn_resnet50_fpn(weights="DEFAULT")
    
    # Custom Anchors for Particles
    anchor_sizes = ((16,), (32,), (64,), (128,), (256,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * 5
    model.rpn.anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)
    
    # Predictors
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, num_classes)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes)
    
    # Increase detection capacity for crowded images
    model.roi_heads.detections_per_img = 400 

    train(model, train_loader, val_loader, device, epochs=200)
    
    save_path = os.path.join(base_dir, "mask_rcnn_particle_model.pth")
    torch.save(model.state_dict(), save_path)
    print(f"Model saved successfully at: {save_path}")

if __name__ == "__main__":
    multiprocessing.freeze_support()
    main()



Here is a step-by-step breakdown of exactly what this code does, with a special focus on the data augmentation and training pipeline.

### 1. Data Augmentation (`get_train_transforms`)

Data augmentation artificially expands your dataset by altering the images during training. This forces the model to learn the actual features of a particle rather than memorizing the specific training images, which prevents overfitting.

Here is what your Albumentations pipeline is doing:

* **PadIfNeeded:** Mask R-CNN uses a Feature Pyramid Network (FPN) that requires image dimensions to be divisible by 32. This pads the edges with black pixels to strictly enforce that rule without warping the image.
* **Spatial Augmentations (Flips & Rotations):** `HorizontalFlip`, `VerticalFlip`, and `RandomRotate90` randomly flip and rotate the image. Since particles generally look like particles regardless of their orientation, this is a perfect way to multiply your training data.
* **Affine Transformations:** The `Affine` block randomly translates (shifts) the image by up to 10%, scales it between 85% and 115%, and rotates it between -30 and 30 degrees.
* **Pixel-Level Augmentations (The `OneOf` Block):** This block chooses *only one* of the following effects to apply 40% of the time:
* *Brightness/Contrast:* Simulates different microscope lighting conditions.
* *HueSaturationValue:* Alters the color slightly.
* *GaussNoise:* Adds static/grain, which helps the model learn to ignore camera sensor noise.


* **Normalize & ToTensorV2:** Converts the image colors to standard ImageNet statistics (required since you are using a pre-trained ResNet50 backbone) and converts the numpy arrays into PyTorch tensors.
* **BboxParams:** Ensures that if an augmentation pushes a particle almost entirely off-screen, the bounding box is dropped if its area is less than 8 pixels or visibility drops below 10%.

### 2. Dataset Parsing (`CocoDataset`)

This class bridges the gap between your JSON file and PyTorch.

* It reads the `sustain_fixed.json` COCO file.
* It looks at the `segmentation` polygons (a list of X, Y coordinates) and uses OpenCV (`cv2.fillPoly`) to draw filled shapes, creating the binary, pixel-perfect masks the model needs to learn from.

### 3. Model Customization

You aren't just using an out-of-the-box model; you have specifically tuned it for particle analysis:

* **Anchor Generator:** Anchors are the initial "guess" boxes the model places over the image. You provided custom sizes (`16` to `256`) and ratios (`0.5`, `1.0`, `2.0`). This tells the model to look for objects ranging from tiny to large, and from elongated to perfectly square.
* **detections_per_img = 400:** By default, Mask R-CNN stops looking after 100 objects. You increased this to 400, which is critical for crowded microscopic images where hundreds of particles might be present.

### 4. The Training Engine (`train` function)

This is where the actual learning happens over the course of 200 maximum epochs.

* **Optimizer (AdamW):** Updates the model weights. AdamW is excellent for computer vision tasks as it includes weight decay, which acts as a regularizer to keep weights from exploding.
* **Learning Rate Scheduler (`ReduceLROnPlateau`):** Starts the learning rate at `1e-4`. If the model's accuracy (Val IoU) stops improving for 10 epochs, it cuts the learning rate in half (`factor=0.5`). This allows the model to take smaller, more precise learning steps as it gets closer to the optimal solution.
* **Gradient Clipping (`clip_grad_norm_`):** Sometimes, a weird batch of images can cause a massive mathematical error (exploding gradients) that ruins the model's progress. This limits the maximum update size to `1.0` to keep training stable.
* **Early Stopping:** Evaluates the model on the validation set using **Intersection over Union (IoU)**, which measures how perfectly the predicted mask overlaps the ground truth mask. If the IoU doesn't improve by at least 0.001 for 35 straight epochs, training stops early to save time and prevent over-training.

